# 面试问题：HyDE 为什么能跨越词汇鸿沟，怎样与关键词检索通过 RRF 融合并保持可追溯？

        ## 可直接复述的回答主线

        1. HyDE 先根据用户问题生成一段假想答案文档，再用这段文档的语义表示检索真实知识。
2. 它能把口语表达映射到知识库术语，但假想文档本身不是证据，最终回答只能引用真实文档。
3. 关键词排名和 HyDE 语义排名可以用 Reciprocal Rank Fusion 合并，不需要直接比较两种分数尺度。
4. 底层实现应展示原查询、假想文档、两路排名、每项 RRF 贡献和最终证据。
5. 歧义问题可能生成错误假想文档，因此需要意图约束、来源过滤和原查询信号兜底。
6. 生产系统还需真实 embedding、生成缓存、查询注入防护、重排、引用核验和版本监控。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例包含五条口语化客服问题和六篇规范术语知识文档。问题故意不使用“退款、自动续费、发票、物流、账户”等文档词，形成词汇鸿沟；另有一条“账户冻结”歧义失败探针。

In [1]:
import math  # 计算向量范数和余弦相似度。
import re  # 对空格知识词和连续问题做轻量切分。
documents = [{"id": "doc-renew", "text": "会员 自动 续费 可以 在 设置 页面 关闭"}, {"id": "doc-refund", "text": "退款 审核 通过 后 三个 工作日 到账"}, {"id": "doc-invoice", "text": "电子 发票 抬头 在 开票 前 可以 修改"}, {"id": "doc-logistics", "text": "物流 四十八 小时 未更新 可以 提交 催件"}, {"id": "doc-account", "text": "账户 锁定 后 完成 实名 验证 可以 重置"}, {"id": "doc-security", "text": "客服 不会 索取 密码 或 验证码"}]  # 定义六篇使用规范业务术语的知识文档。
queries = [{"id": "hyde-01", "question": "怎么停掉会员扣款", "expected": "doc-renew"}, {"id": "hyde-02", "question": "钱怎么还没回来", "expected": "doc-refund"}, {"id": "hyde-03", "question": "票据公司名最晚何时改", "expected": "doc-invoice"}, {"id": "hyde-04", "question": "包裹两天都不动怎么办", "expected": "doc-logistics"}, {"id": "hyde-05", "question": "账号进不去怎么恢复", "expected": "doc-account"}]  # 定义五条与知识库存在词汇鸿沟的口语查询。
canonical_terms = ("续费", "关闭", "退款", "到账", "发票", "抬头", "物流", "催件", "账户", "锁定", "密码", "验证码")  # 定义教学语义向量的十二个可解释维度。
hypotheses = {"hyde-01": "会员 自动 续费 在 设置 页面 关闭", "hyde-02": "退款 审核 后 等待 到账", "hyde-03": "电子 发票 抬头 在 开票 前 修改", "hyde-04": "物流 长时间 未更新 需要 催件", "hyde-05": "账户 锁定 后 进行 验证 重置"}  # 为五条口语问题构造规范术语假想文档。
def word_tokens(text):  # 提取空格词和连续中文片段供关键词基线。
    return set(re.findall(r"[A-Za-z0-9]+|[\u4e00-\u9fff]+", text.lower().replace(" ", "\n")))  # 返回严格词级集合以保留词汇鸿沟。
print("教学实验输入：五条口语查询与六篇规范文档")  # 标记下方为离线检索案例。
for query in queries:  # 逐条展示原问题、假想文档和期望证据。
    print(f"{query['id']} expected={query['expected']:<13} query={query['question']} | HyDE={hypotheses[query['id']]}")  # 输出当前查询的两种文本。
print("文档：", [(document["id"], document["text"]) for document in documents])  # 展示真实可引用知识。

教学实验输入：五条口语查询与六篇规范文档
hyde-01 expected=doc-renew     query=怎么停掉会员扣款 | HyDE=会员 自动 续费 在 设置 页面 关闭
hyde-02 expected=doc-refund    query=钱怎么还没回来 | HyDE=退款 审核 后 等待 到账
hyde-03 expected=doc-invoice   query=票据公司名最晚何时改 | HyDE=电子 发票 抬头 在 开票 前 修改
hyde-04 expected=doc-logistics query=包裹两天都不动怎么办 | HyDE=物流 长时间 未更新 需要 催件
hyde-05 expected=doc-account   query=账号进不去怎么恢复 | HyDE=账户 锁定 后 进行 验证 重置
文档： [('doc-renew', '会员 自动 续费 可以 在 设置 页面 关闭'), ('doc-refund', '退款 审核 通过 后 三个 工作日 到账'), ('doc-invoice', '电子 发票 抬头 在 开票 前 可以 修改'), ('doc-logistics', '物流 四十八 小时 未更新 可以 提交 催件'), ('doc-account', '账户 锁定 后 完成 实名 验证 可以 重置'), ('doc-security', '客服 不会 索取 密码 或 验证码')]


## 2. Baseline / 基线：只用原查询做严格关键词 overlap

原问题没有空格且使用“扣款、钱回来、票据、包裹、账号”等口语，严格词级 overlap 大量为零。零分时稳定 ID 排序会产生看似确定但无依据的 top-1。

In [2]:
def lexical_ranking(question):  # 对原查询和真实文档计算严格词级 overlap。
    query_terms = word_tokens(question)  # 提取原查询词集合。
    rows = []  # 保存六篇文档的关键词得分。
    for document in documents:  # 逐文档计算词集合交集。
        overlap = sorted(query_terms & set(document["text"].split()))  # 只接受完整业务词命中。
        rows.append({"doc": document, "score": len(overlap), "overlap": overlap})  # 保存得分和命中词。
    return sorted(rows, key=lambda row: (-row["score"], row["doc"]["id"]))  # 按得分和ID稳定排序。
baseline_rows = []  # 保存五条口语查询的关键词 top-1。
for query in queries:  # 逐查询执行关键词检索。
    ranking = lexical_ranking(query["question"])  # 获取完整关键词排名。
    top = ranking[0]  # 读取稳定 top-1。
    baseline_rows.append({"id": query["id"], "doc": top["doc"]["id"], "score": top["score"], "correct": top["doc"]["id"] == query["expected"], "ranking": ranking})  # 保存结果和排名。
print("Baseline 原查询关键词排名")  # 标记下表没有 HyDE 扩展。
print("请求      top1          score  correct  overlap")  # 输出基线结果表头。
for row in baseline_rows:  # 逐条展示零分或弱命中。
    print(f"{row['id']:<9} {row['doc']:<13} {row['score']:>5} {str(row['correct']):>8}  {row['ranking'][0]['overlap']}")  # 输出当前查询 top-1。

Baseline 原查询关键词排名
请求      top1          score  correct  overlap
hyde-01   doc-account       0    False  []
hyde-02   doc-account       0    False  []
hyde-03   doc-account       0    False  []
hyde-04   doc-account       0    False  []
hyde-05   doc-account       0     True  []


## 3. 底层实现：可解释语义向量、HyDE 排名与 RRF

教学 embedding 不是现成模型，而是十二维规范概念计数。加权 RRF 仍只融合名次；当关键词 top 分为零时把该无证据通道权重降到 0.25，HyDE 通道保持 1.0，并保存两路贡献。

In [3]:
def concept_vector(text):  # 把文本映射到十二维可解释业务概念向量。
    return [float(term in text) for term in canonical_terms]  # 每维表示规范概念是否出现。
def cosine(left, right):  # 手写两个概念向量的余弦相似度。
    dot = sum(a * b for a, b in zip(left, right))  # 计算向量点积。
    left_norm = math.sqrt(sum(value * value for value in left))  # 计算左向量二范数。
    right_norm = math.sqrt(sum(value * value for value in right))  # 计算右向量二范数。
    return dot / (left_norm * right_norm) if left_norm > 0.0 and right_norm > 0.0 else 0.0  # 对零向量返回零相似度。
def hyde_ranking(hypothesis):  # 用假想文档概念向量检索真实文档。
    query_vector = concept_vector(hypothesis)  # 编码假想答案中的规范概念。
    rows = [{"doc": document, "score": cosine(query_vector, concept_vector(document["text"]))} for document in documents]  # 计算每篇真实文档的余弦相似度。
    return sorted(rows, key=lambda row: (-row["score"], row["doc"]["id"]))  # 按余弦分和ID稳定排序。
def reciprocal_rank_fusion(rankings, k=60):  # 用置信加权名次而非直接比较原始分数融合多路检索。
    scores = {}  # 累积每篇文档的 RRF 分。
    contributions = {}  # 保存每一路对文档的名次贡献。
    for channel, ranking in rankings.items():  # 逐检索通道遍历排名。
        channel_weight = 0.25 if channel == "lexical" and ranking[0]["score"] == 0 else 1.0  # 对完全零命中的关键词通道降低置信权重。
        for rank, row in enumerate(ranking, start=1):  # 从一开始记录文档名次。
            document_id = row["doc"]["id"]  # 读取当前文档唯一标识。
            contribution = channel_weight / (k + rank)  # 计算带通道置信度的倒数名次贡献。
            scores[document_id] = scores.get(document_id, 0.0) + contribution  # 累加跨通道 RRF 分。
            contributions.setdefault(document_id, {})[channel] = contribution  # 保存当前通道分项。
    fused = sorted(scores.items(), key=lambda item: (-item[1], item[0]))  # 按融合分和文档ID排序。
    return fused, contributions  # 返回融合名次和可解释贡献。
first_query = queries[0]  # 选择会员扣款问题展示两路排名。
first_lexical = lexical_ranking(first_query["question"])  # 计算原查询关键词排名。
first_hyde = hyde_ranking(hypotheses[first_query["id"]])  # 计算假想文档语义排名。
first_fused, first_contributions = reciprocal_rank_fusion({"lexical": first_lexical, "hyde": first_hyde})  # 融合两路名次。
print("hyde-01 两路排名与 RRF 前五")  # 标记下表展示融合过程。
print("doc             lexical_rank  hyde_rank  lexical贡献  hyde贡献  RRF")  # 输出 RRF 分项表头。
lexical_positions = {row["doc"]["id"]: rank for rank, row in enumerate(first_lexical, start=1)}  # 建立关键词名次索引。
hyde_positions = {row["doc"]["id"]: rank for rank, row in enumerate(first_hyde, start=1)}  # 建立 HyDE 名次索引。
for document_id, fused_score in first_fused[:5]:  # 展示融合前五篇文档。
    contribution = first_contributions[document_id]  # 读取两路倒数名次贡献。
    print(f"{document_id:<15} {lexical_positions[document_id]:>12} {hyde_positions[document_id]:>10} {contribution['lexical']:>12.5f} {contribution['hyde']:>9.5f} {fused_score:>7.5f}")  # 输出当前文档融合分项。

hyde-01 两路排名与 RRF 前五
doc             lexical_rank  hyde_rank  lexical贡献  hyde贡献  RRF
doc-renew                  5          1      0.00385   0.01639 0.02024
doc-account                1          2      0.00410   0.01613 0.02023
doc-invoice                2          3      0.00403   0.01587 0.01991
doc-logistics              3          4      0.00397   0.01562 0.01959
doc-refund                 4          5      0.00391   0.01538 0.01929


## 4. 结果表与结果解读

对五条问题使用相同 HyDE 规则和 RRF 参数。最终答案只引用真实 doc ID，假想文档仅用于生成检索向量，不能作为 citation。

In [4]:
fused_rows = []  # 保存五条查询的 HyDE 与 RRF 结果。
for query in queries:  # 逐查询执行两路检索和融合。
    lexical = lexical_ranking(query["question"])  # 获取原查询关键词排名。
    hyde = hyde_ranking(hypotheses[query["id"]])  # 获取假想文档语义排名。
    fused, contributions = reciprocal_rank_fusion({"lexical": lexical, "hyde": hyde})  # 融合两路名次。
    top_document = fused[0][0]  # 读取最终最高 RRF 文档。
    fused_rows.append({"id": query["id"], "lexical_top": lexical[0]["doc"]["id"], "hyde_top": hyde[0]["doc"]["id"], "fused_top": top_document, "correct": top_document == query["expected"], "score": fused[0][1]})  # 保存逐查询证据选择。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(queries)  # 计算原查询关键词 top-1 准确率。
fused_accuracy = sum(row["correct"] for row in fused_rows) / len(queries)  # 计算 HyDE+RRF 证据准确率。
print("请求      lexical_top   HyDE_top      RRF_top       expected      correct")  # 输出逐查询检索对照表头。
for row, query in zip(fused_rows, queries):  # 逐条展示三种 top-1 与期望证据。
    print(f"{row['id']:<9} {row['lexical_top']:<13} {row['hyde_top']:<13} {row['fused_top']:<13} {query['expected']:<13} {str(row['correct']):>7}")  # 输出当前查询结果。
print(f"结果解读：原查询关键词准确率={baseline_accuracy:.1%}，HyDE+RRF={fused_accuracy:.1%}；假想文本没有进入最终证据。")  # 解释词汇鸿沟和 grounding 边界。

请求      lexical_top   HyDE_top      RRF_top       expected      correct
hyde-01   doc-account   doc-renew     doc-renew     doc-renew        True
hyde-02   doc-account   doc-refund    doc-refund    doc-refund       True
hyde-03   doc-account   doc-invoice   doc-invoice   doc-invoice      True
hyde-04   doc-account   doc-logistics doc-logistics doc-logistics    True
hyde-05   doc-account   doc-account   doc-account   doc-account      True
结果解读：原查询关键词准确率=20.0%，HyDE+RRF=100.0%；假想文本没有进入最终证据。


## 5. 失败案例与修正

“账户冻结可以提现吗”中的“冻结”可能被错误解释为退款冻结。错误 HyDE 会把 doc-refund 排首；加入显式实体“账户”约束后改写为账户锁定恢复。

In [5]:
ambiguous_query = "账户冻结可以提现吗"  # 构造具有账户与资金双重含义的歧义问题。
wrong_hypothesis = "退款 冻结 资金 等待 到账"  # 模拟没有实体约束的错误假想文档。
constrained_hypothesis = "账户 锁定 后 完成 验证 重置"  # 根据原查询显式账户实体约束假想文档。
wrong_top = hyde_ranking(wrong_hypothesis)[0]["doc"]["id"]  # 读取错误 HyDE 的最高真实文档。
fixed_top = hyde_ranking(constrained_hypothesis)[0]["doc"]["id"]  # 读取实体约束后的最高真实文档。
print(f"错误行为：query={ambiguous_query}，HyDE={wrong_hypothesis}，top={wrong_top}")  # 展示假想文档偏航后的错误检索。
print(f"修正行为：识别实体=账户，HyDE={constrained_hypothesis}，top={fixed_top}")  # 展示受控假想文档和正确证据。

错误行为：query=账户冻结可以提现吗，HyDE=退款 冻结 资金 等待 到账，top=doc-refund
修正行为：识别实体=账户，HyDE=账户 锁定 后 完成 验证 重置，top=doc-account


## 6. 生产边界

十二维概念向量只用于解释机制。生产中应使用版本化 embedding 与生成模型，并监控 HyDE 幻觉、缓存命中、两路排名分歧、查询注入、租户 ACL、重排延迟和 citation 支持度。

In [6]:
rank_disagreement = sum(row["lexical_top"] != row["hyde_top"] for row in fused_rows) / len(fused_rows)  # 统计关键词与 HyDE top-1 分歧率。
diagnostics = {"queries": len(queries), "baseline_accuracy": baseline_accuracy, "fused_accuracy": fused_accuracy, "rank_disagreement": rank_disagreement, "hypotheses_used_as_citations": 0}  # 汇总检索质量和 grounding 指标。
print("生产监控快照：", diagnostics)  # 输出 HyDE+RRF 服务需要持续观察的信号。

生产监控快照： {'queries': 5, 'baseline_accuracy': 0.2, 'fused_accuracy': 1.0, 'rank_disagreement': 0.8, 'hypotheses_used_as_citations': 0}


## 7. 最小回归测试

只验证样本规模、RRF 贡献、词汇鸿沟改善、最终 grounding 和歧义修正。

In [7]:
assert len(queries) >= 5 and len(documents) >= 5  # 保证案例包含足够查询和真实文档。
assert all(abs(sum(first_contributions[document_id].values()) - score) < 1.0e-12 for document_id, score in first_fused)  # 保证 RRF 总分等于两路贡献和。
assert fused_accuracy > baseline_accuracy  # 保证同一批口语查询上的融合结果优于关键词基线。
assert all(row["fused_top"].startswith("doc-") for row in fused_rows)  # 保证最终证据始终来自真实知识文档而非假想文本。
assert wrong_top == "doc-refund" and fixed_top == "doc-account"  # 保证歧义失败和实体约束修正均真实发生。